# 04 - Portfolio Optimization

Find optimal portfolio weights. Choose your optimization target from the available strategies.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from src.data import fetch_prices, compute_returns
from src.optimization import (
    optimize_portfolio, optimize_all_strategies,
    black_litterman_optimize, kelly_criterion,
    efficient_frontier_data, hierarchical_risk_parity,
    compute_expected_returns, compute_covariance,
    OPTIMIZATION_TARGETS,
)
from src.portfolio import build_portfolio_returns, compare_portfolios
from src.metrics import metrics_table

In [ ]:
tickers = ["AAPL", "MSFT", "VWCE.MI", "SXR8.DE", "BTC-USD"]
start_date = "2018-01-01"
end_date = "2025-01-01"

prices = fetch_prices(tickers, start=start_date, end=end_date)
returns = compute_returns(prices)
print(f"Loaded {len(prices)} trading days for {len(tickers)} assets")

## Available Optimization Targets

| Target | Description |
|--------|-------------|
| `max_sharpe` | Maximize Sharpe ratio |
| `min_volatility` | Minimize portfolio volatility |
| `max_quadratic_utility` | Maximize quadratic utility (risk-averse) |
| `efficient_return` | Minimize risk for a given target return |
| `efficient_risk` | Maximize return for a given target risk |
| `min_cvar` | Minimize Conditional Value-at-Risk |
| `min_semivariance` | Minimize semivariance (downside risk) |
| `max_sortino` | Maximize Sortino ratio |
| `max_return_min_risk` | Max Sharpe with L2 regularization (more diversified) |
| `hrp` | Hierarchical Risk Parity |

## Choose Your Optimization Target

Change `target` below to any value from the table above.

In [ ]:
target = "max_sharpe"

target_value = None
# For 'efficient_return': set target annual return, e.g. 0.15
# For 'efficient_risk': set target annual volatility, e.g. 0.20

weights = optimize_portfolio(prices, target=target, target_value=target_value)
print(f"Optimization target: {target}")
print("Optimal Weights:")
for t, w in sorted(weights.items(), key=lambda x: -x[1]):
    if abs(w) > 0.001:
        print(f"  {t}: {w:.2%}")

In [ ]:
target = "min_cvar"
weights_cvar = optimize_portfolio(prices, target=target)
print(f"Optimization target: {target}")
print("Optimal Weights:")
for t, w in sorted(weights_cvar.items(), key=lambda x: -x[1]):
    if abs(w) > 0.001:
        print(f"  {t}: {w:.2%}")

In [ ]:
target = "max_sortino"
weights_sortino = optimize_portfolio(prices, target=target)
print(f"Optimization target: {target}")
print("Optimal Weights:")
for t, w in sorted(weights_sortino.items(), key=lambda x: -x[1]):
    if abs(w) > 0.001:
        print(f"  {t}: {w:.2%}")

## Compare All Optimization Strategies

In [ ]:
all_strategies = optimize_all_strategies(prices)

fig = go.Figure()
for name, weights in all_strategies.items():
    sorted_w = sorted(weights.items(), key=lambda x: x[0])
    fig.add_trace(go.Bar(
        x=[t for t, _ in sorted_w],
        y=[w * 100 for _, w in sorted_w],
        name=name,
    ))
fig.update_layout(
    title='Optimal Weights by Strategy (%)',
    yaxis_title='Weight %',
    barmode='group',
    template='plotly_white',
    height=600,
)
fig.show()

## Efficient Frontier

In [ ]:
frontier_rets, frontier_risks, max_sharpe_pt, min_vol_pt = efficient_frontier_data(prices)
mu = compute_expected_returns(prices)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=frontier_risks, y=frontier_rets,
    mode='lines', name='Efficient Frontier',
    line=dict(width=3, color='blue'),
))

fig.add_trace(go.Scatter(
    x=[max_sharpe_pt['risk']], y=[max_sharpe_pt['return']],
    mode='markers+text', name=f"Max Sharpe ({max_sharpe_pt['sharpe']:.2f})",
    marker=dict(size=14, color='green', symbol='star'),
    text=['Max Sharpe'], textposition='top center',
))

fig.add_trace(go.Scatter(
    x=[min_vol_pt['risk']], y=[min_vol_pt['return']],
    mode='markers+text', name="Min Volatility",
    marker=dict(size=14, color='red', symbol='diamond'),
    text=['Min Vol'], textposition='top center',
))

for t in prices.columns:
    ann_vol = returns[t].std() * np.sqrt(252)
    ann_ret = mu.get(t, 0)
    fig.add_trace(go.Scatter(
        x=[ann_vol], y=[ann_ret], mode='markers+text', name=t,
        text=[t], textposition='top center', marker=dict(size=10),
    ))

fig.update_layout(
    title='Efficient Frontier', xaxis_title='Annualized Volatility',
    yaxis_title='Annualized Expected Return', template='plotly_white', height=700,
)
fig.show()

## Black-Litterman Model

Incorporate your own market views. Set expected returns for specific assets.

In [ ]:
views = {
    "AAPL": 0.15,
    "BTC-USD": 0.25,
    "MSFT": 0.12,
}

view_confidences = [0.05, 0.10, 0.04]

market_prior = {t: 1.0/len(tickers) for t in tickers}

bl_weights = black_litterman_optimize(
    prices,
    views=views,
    view_confidences=view_confidences,
    market_prior=market_prior,
    target="max_sharpe",
)

print("Black-Litterman Weights:")
for t, w in sorted(bl_weights.items(), key=lambda x: -x[1]):
    if abs(w) > 0.001:
        print(f"  {t}: {w:.2%}")

## Kelly Criterion

In [ ]:
kelly_w = kelly_criterion(prices)
print("Kelly Criterion Weights:")
for t, w in sorted(kelly_w.items(), key=lambda x: -x[1]):
    print(f"  {t}: {w:.2%}")

## Backtest All Optimized Portfolios

In [ ]:
optimized = {}
for name, w in all_strategies.items():
    optimized[name] = build_portfolio_returns(prices, w)
optimized['Black-Litterman'] = build_portfolio_returns(prices, bl_weights)
optimized['Kelly'] = build_portfolio_returns(prices, kelly_w)
optimized['Equal Weight'] = build_portfolio_returns(prices, {t: 1/len(tickers) for t in tickers})

opt_metrics = metrics_table(optimized)
opt_metrics.style.format('{:.4f}').set_caption('All Strategies Compared')

In [ ]:
curves = compare_portfolios(optimized, initial_value=10000)

fig = go.Figure()
for col in curves.columns:
    fig.add_trace(go.Scatter(x=curves.index, y=curves[col], name=col, mode='lines'))
fig.update_layout(
    title='All Strategies Equity Curve (Initial: $10,000)',
    yaxis_title='Portfolio Value ($)', xaxis_title='Date',
    template='plotly_white', height=600,
)
fig.show()